# W261 Final Project Phase II - OTPW EDA
**Team:** Section 3, Group 2

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

spark.conf.set("spark.sql.ansi.enabled", "false")

# Data Preparation
## Load OTPW and Save to Parquet

Where are data saved?

In [0]:
# display(dbutils.fs.ls("dbfs:/mnt/mids-w261/"))
# display(dbutils.fs.ls("dbfs:/mnt/mids-w261/OTPW_12M/"))
display(dbutils.fs.ls("dbfs:/mnt/mids-w261/OTPW_12M/OTPW_12M/"))

In [0]:
# Always read from parquet

# dbutils.fs.mkdirs(TEAM_DIR)
TEAM_DIR = "dbfs:/student-groups/Group_3_2"

Load 3 Months OTPW

In [0]:
# Checkpoint as parquet (run once, then comment out)

# df_otpw_3m = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")
# df_otpw_3m.write.mode("overwrite").parquet(f"{TEAM_DIR}/otpw_3m.parquet")
# print("Parquet saved.")

df_3m = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")

Load 12 Months OTPW

In [0]:
# df_otpw_12m = spark.read.format("csv").option("header","true").load("dbfs:/mnt/mids-w261/OTPW_12M/OTPW_12M/OTPW_12M_2015.csv.gz")
# df_otpw_12m.write.mode("overwrite").parquet(f"{TEAM_DIR}/otpw_12m.parquet")
# print("Parquet saved.")

df_12m = spark.read.parquet(f"{TEAM_DIR}/otpw_12m.parquet")

In [0]:
# View distinct flight dates to verify that 12m dataset is from 2015
display(df_12m.select("FL_DATE").distinct())

In [0]:
# Dataset size of 12m otpw
num_rows = df_12m.count()
num_cols = len(df_12m.columns)
print(f"Rows: {num_rows:,}")
print(f"Columns: {num_cols}")

In [0]:
display(df_12m)

In [0]:
display(df_12m)

# EDA
## Dataset Overview

In [0]:
rows_3m = df_3m.count()
cols_3m = len(df_3m.columns)
rows_12m = df_12m.count()
cols_12m = len(df_12m.columns)

print(f"3-Month OTPW:  {rows_3m:,} rows x {cols_3m} columns")
print(f"12-Month OTPW: {rows_12m:,} rows x {cols_12m} columns")

In [0]:
# Full schema
df_12m.printSchema()

## Missing Value Analysis

In [0]:
null_exprs_3m = [
    count(when(col(c).isNull() | (col(c).cast("string") == ""), c)).alias(c)
    for c in df_3m.columns
]
null_counts_3m = df_3m.select(null_exprs_3m).toPandas().T
null_counts_3m.columns = ["null_count"]
null_counts_3m["null_pct"] = (null_counts_3m["null_count"] / rows_3m * 100).round(2)
null_counts_3m = null_counts_3m.reset_index().rename(columns={"index": "column_name"}).sort_values("null_pct", ascending=False)

high_missing_3m = null_counts_3m[null_counts_3m["null_pct"] > 50]
print(f"3-Month — Columns with >50% missing: {len(high_missing_3m)}")
display(high_missing_3m)

In [0]:
null_exprs_12m = [
    count(when(col(c).isNull() | (col(c).cast("string") == ""), c)).alias(c)
    for c in df_12m.columns
]
null_counts_12m = df_12m.select(null_exprs_12m).toPandas().T
null_counts_12m.columns = ["null_count"]
null_counts_12m["null_pct"] = (null_counts_12m["null_count"] / rows_12m * 100).round(2)
null_counts_12m = null_counts_12m.reset_index().rename(columns={"index": "column_name"}).sort_values("null_pct", ascending=False)

high_missing_12m = null_counts_12m[null_counts_12m["null_pct"] > 50]
print(f"12-Month — Columns with >50% missing: {len(high_missing_12m)}")
display(high_missing_12m)

## REPORT_TYPE Filtering

The OTPW dataset joins flight records with weather observations. However, the weather data contains multiple observation types, hourly surface reports, daily summaries, monthly summaries, and special observations, each with different levels of completeness. The high missing rates above suggest that many weather columns are only populated for certain report types.

The `REPORT_TYPE` column identifies the type of weather observation. We investigate its distribution to determine whether filtering to a single, well-populated report type can reduce missing data in the hourly weather features we plan to use as model inputs.

Of all flights, **87.5% are matched to FM-15** records - the standard hourly aviation surface observation reported by airports. The remaining 12.5% are matched to other report types (SOD, SOM, SHEF, etc.) which are daily/monthly summaries and do not populate the hourly weather columns.

FM-15 records show dramatically lower null rates for key hourly weather features compared to the full dataset. Since the non-FM-15 rows carry mostly empty weather data, they would contribute noise**For modeling, we will filter to FM-15 rows only**.

In [0]:
print("REPORT_TYPE distribution (12-Month):")
df_12m.groupBy("REPORT_TYPE").count().orderBy(desc("count")).show(10)

df_fm15_12m = df_12m.filter(col("REPORT_TYPE") == "FM-15")
fm15_count = df_fm15_12m.count()
print(f"\nFM-15 rows: {fm15_count:,} out of {rows_12m:,} total ({fm15_count/rows_12m*100:.1f}%)")

## Duplicate Check

In [0]:
distinct_3m = df_3m.dropDuplicates().count()
distinct_12m = df_12m.dropDuplicates().count()

print(f"3-Month  — Total: {rows_3m:,}  Distinct: {distinct_3m:,}  Duplicates: {rows_3m - distinct_3m:,}")
print(f"12-Month — Total: {rows_12m:,}  Distinct: {distinct_12m:,}  Duplicates: {rows_12m - distinct_12m:,}")

## Target Variable Distribution

A cancelled flight didn't depart so we cannot predict the delay of something that never happened.

In [0]:
df_3m.groupBy("CANCELLED").count().show()
df_12m.groupBy("CANCELLED").count().show()

In [0]:
cancelled_3m = df_3m.filter(col("CANCELLED") == 1)
total_cancelled_3m = cancelled_3m.count()
cancelled_with_del15_3m = cancelled_3m.filter(col('DEP_DEL15').isNotNull()).count()

cancelled_12m = df_12m.filter(col("CANCELLED") == 1)
total_cancelled_12m = cancelled_12m.count()
cancelled_with_del15_12m = cancelled_12m.filter(col('DEP_DEL15').isNotNull()).count()

print(f"3-Month  — Cancelled: {total_cancelled_3m} ({total_cancelled_3m / rows_3m * 100:.2f}%)  with DEP_DEL15: {cancelled_with_del15_3m}")
print(f"12-Month — Cancelled: {total_cancelled_12m} ({total_cancelled_12m / rows_12m * 100:.2f}%)  with DEP_DEL15: {cancelled_with_del15_12m}")

In [0]:
df_3m = df_3m.filter(col("CANCELLED") == 0)
df_12m = df_12m.filter(col("CANCELLED") == 0)
rows_3m = df_3m.count()
rows_12m = df_12m.count()
print(f"After removing cancelled flights:")
print(f"  3-Month:  {rows_3m:,} rows")
print(f"  12-Month: {rows_12m:,} rows")

In [0]:
target_3m = df_3m.groupBy("DEP_DEL15").count().toPandas()
target_3m["pct"] = (target_3m["count"] / target_3m["count"].sum() * 100).round(2)
target_3m = target_3m.sort_values("DEP_DEL15")

target_12m = df_12m.groupBy("DEP_DEL15").count().toPandas()
target_12m["pct"] = (target_12m["count"] / target_12m["count"].sum() * 100).round(2)
target_12m = target_12m.sort_values("DEP_DEL15")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, tdf, title in zip(axes, [target_3m, target_12m], ["3-Month OTPW", "12-Month OTPW"]):
    bars = ax.bar(tdf["DEP_DEL15"].astype(str), tdf["count"])
    ax.set_title(f"DEP_DEL15 Distribution ({title})")
    ax.set_xlabel("DEP_DEL15 (0=On-time, 1=Delayed ≥15min)")
    ax.set_ylabel("Count")
    for i, (_, row) in enumerate(tdf.iterrows()):
        ax.text(i, row["count"], f'{row["pct"]}%', ha="center", va="bottom")
plt.tight_layout()
plt.show()

The class distribution is consistent across both 3-month and 12-month datasets at roughly 80/20 (on-time vs delayed). This imbalance means a naive model predicting "on-time" for every flight would achieve ~80% accuracy while being entirely useless for identifying delays. For modeling, we will prioritize recall rather than accuracy.

## Numeric Feature Statistics

In [0]:
numeric_cols = [
    "DISTANCE", "CRS_ELAPSED_TIME", "CRS_DEP_TIME",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyWindSpeed", "HourlyWindDirection",
    "HourlyVisibility", "HourlyPrecipitation",
    "HourlyRelativeHumidity", "HourlyStationPressure"
]

print("3-Month:")
df_num_3m = df_3m.select([col(c).cast("double").alias(c) for c in numeric_cols if c in df_3m.columns])
display(df_num_3m.describe())

print("\n12-Month:")
df_num_12m = df_12m.select([col(c).cast("double").alias(c) for c in numeric_cols if c in df_12m.columns])
display(df_num_12m.describe())

The summary statistics confirm that the key numeric features have majority coverage (high counts relative to dataset size). The min/max ranges are reasonable, no obviously broken values. Flight features like DISTANCE and CRS_ELAPSED_TIME show expected ranges. Weather features show typical variability: temperatures from cold to hot seasons, wind speeds from moderate with occasional extremes, and precipitation heavily concentrated near zero (most hours have no rain).

## Categorical Features

In [0]:
cat_cols = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "ORIGIN_STATE_ABR", "DAY_OF_WEEK", "MONTH", "QUARTER", "DAY_OF_MONTH"]

print("Categorical feature cardinality:")
print(f"{'Feature':<25} {'3-Month':>10} {'12-Month':>10}")
print("-" * 47)
for c in cat_cols:
    n_3m = df_3m.select(c).distinct().count()
    n_12m = df_12m.select(c).distinct().count()
    print(f"{c:<25} {n_3m:>10} {n_12m:>10}")

## Delay Patterns

We analyze delay rates across temporal and operational dimensions to identify patterns that may be predictive of `DEP_DEL15`. Understanding when and where delays concentrate helps inform feature selection and provides operational context for the classification task.

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

dow = df_12m.groupBy("DAY_OF_WEEK").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .orderBy("DAY_OF_WEEK").toPandas()
axes[0].bar(dow["DAY_OF_WEEK"].astype(str), dow["delay_rate"])
axes[0].set_title("Delay Rate by Day of Week (12M)")
axes[0].set_xlabel("Day of Week (1=Mon, 7=Sun)")
axes[0].set_ylabel("Delay Rate (%)")

df_hour = df_12m.withColumn("DEP_HOUR", (col("CRS_DEP_TIME").cast("int") / 100).cast("int"))
hourly = df_hour.groupBy("DEP_HOUR").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .orderBy("DEP_HOUR").toPandas()
axes[1].bar(hourly["DEP_HOUR"], hourly["delay_rate"])
axes[1].set_title("Delay Rate by Hour of Day (12M)")
axes[1].set_xlabel("Scheduled Departure Hour")
axes[1].set_ylabel("Delay Rate (%)")
axes[1].set_xticks(range(0, 24))

monthly = df_12m.groupBy("MONTH").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .orderBy("MONTH").toPandas()
axes[2].bar(monthly["MONTH"].astype(str), monthly["delay_rate"])
axes[2].set_title("Delay Rate by Month (12M)")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Delay Rate (%)")

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

carrier = df_12m.groupBy("OP_UNIQUE_CARRIER").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .orderBy(desc("delay_rate")).toPandas()
axes[0].barh(carrier["OP_UNIQUE_CARRIER"], carrier["delay_rate"])
axes[0].set_title("Delay Rate by Carrier (12M)")
axes[0].set_xlabel("Delay Rate (%)")
axes[0].set_ylabel("Carrier")
axes[0].invert_yaxis()

airport = df_12m.groupBy("ORIGIN").agg(
    count("*").alias("total_flights"),
    spark_sum(col("DEP_DEL15").cast("int")).alias("delayed")
).withColumn("delay_rate", (col("delayed") / col("total_flights") * 100).cast("double")) \
 .filter(col("total_flights") >= 1000) \
 .orderBy(desc("delay_rate")).limit(15).toPandas()
axes[1].barh(airport["ORIGIN"], airport["delay_rate"])
axes[1].set_title("Top 15 Airports by Delay Rate (12M, ≥1000 flights)")
axes[1].set_xlabel("Delay Rate (%)")
axes[1].set_ylabel("Origin Airport")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

Delay rates increases throughout the day, peaking in evening hours. This reflects cascading delays as aircraft accumulate lateness across rotations. Monthly patterns show elevated delays during summer (thunderstorm season) and winter (snow/ice). These patterns confirm that time-of-day and seasonality are predictive signals.

There is significant variation in delay rates across carriers and airports. Some carriers consistently underperform, and certain hub airports with high traffic volume experience more delays. This validates that OP_UNIQUE_CARRIER and ORIGIN are important categorical features for the model.

## Weather Feature Distributions

In [0]:
weather_cols = ["HourlyVisibility", "HourlyWindSpeed", "HourlyWindGustSpeed", "HourlyPrecipitation", "HourlyDryBulbTemperature"]
existing_weather = [c for c in weather_cols if c in df_12m.columns]

df_weather_eda = df_12m.select([col(c).cast("double").alias(c) for c in existing_weather])
if "HourlyWindSpeed" in existing_weather:
    df_weather_eda = df_weather_eda.filter(
        (col("HourlyWindSpeed").isNull()) | (col("HourlyWindSpeed") < 200)
    )

weather_pd = df_weather_eda.toPandas()

fig, axes = plt.subplots(1, len(existing_weather), figsize=(4 * len(existing_weather), 5))
if len(existing_weather) == 1:
    axes = [axes]
for ax, c in zip(axes, existing_weather):
    weather_pd[c].dropna().plot.box(ax=ax)
    ax.set_title(c)
    ax.set_ylabel("Value")
plt.suptitle("Weather Feature Distributions (12-Month OTPW)", y=1.02)
plt.tight_layout()
plt.show()

The boxplots show that most weather conditions are normal, clear visibility, light winds, no precipitation. However, the outliers (low visibility, high wind, heavy precipitation) represent the bad conditions most likely associated with delays. The long tails in precipitation and wind gust speed are particularly notable since these extreme events.

The delayed-vs-on-time comparison table below quantifies whether weather conditions at the time of departure systematically differ between delayed and on-time flights. Positive differences indicate the value is higher for delayed flights; negative differences indicate it is lower.

In [0]:
weather_compare_cols = [
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyWindSpeed", "HourlyWindGustSpeed", "HourlyVisibility",
    "HourlyPrecipitation", "HourlyRelativeHumidity",
    "HourlyStationPressure", "HourlySeaLevelPressure"
]
existing_wc = [c for c in weather_compare_cols if c in df_12m.columns]

print("Mean Weather Values: Delayed vs On-Time Flights (12-Month)\n")
print(f"{'Feature':<30} {'On-Time':>10} {'Delayed':>10} {'% Diff':>10}")
print("-" * 62)
for c in existing_wc:
    ontime_val = df_12m.filter(col("DEP_DEL15") == 0).agg(avg(col(c)).alias("m")).collect()[0]["m"]
    delayed_val = df_12m.filter(col("DEP_DEL15") == 1).agg(avg(col(c)).alias("m")).collect()[0]["m"]
    if ontime_val is not None and delayed_val is not None and ontime_val != 0:
        pct_diff = (delayed_val - ontime_val) / abs(ontime_val) * 100
        print(f"{c:<30} {ontime_val:>10.3f} {delayed_val:>10.3f} {pct_diff:>+9.2f}%")

### Weather Comparison Observations

- **HourlyPrecipitation (+121.73%):** Delayed flights see more than double the average precipitation, this confirms precipitation is a strong delay signal.
- **HourlyRelativeHumidity (+6.19%):** Delayed flights experience notably higher humidity (63.2% vs 59.5%), associated with fog, low cloud ceilings, and reduced airport capacity.
- **HourlyWindSpeed (+3.96%) / HourlyWindGustSpeed (+3.04%):** Both higher for delayed flights. Stronger sustained winds and gusts reduce runway throughput and slow ground operations.
- **HourlyDryBulbTemperature (-2.77%):** Delayed flights occur at cooler temperatures on average (62.8°F vs 64.5°F), reflecting the impact of winter weather events (snow, ice, deicing operations).
- **HourlyVisibility (-2.28%):** Lower for delayed flights (9.24 vs 9.45 miles).
- **HourlyDewPointTemperature (+0.12%):** Almost same.
- **HourlyStationPressure (-0.00%) / HourlySeaLevelPressure (-0.05%):** Almost same.

**Takeaway:** Precipitation, humidity, wind, visibility, and temperature all show meaningful percentage shifts between delayed and on-time flights. These weather features will be included in the modeling pipeline. Pressure features show minimal difference. Likely will use it for feature engineering if necessary.

## Data Leakage Check

Columns that are only known after the flight departs must be excluded from features. Using them would leak future information into the model.

In [0]:
leakage_candidates = [
    "DEP_TIME", "DEP_DELAY", "DEP_DEL15", "DEP_DELAY_NEW", "DEP_DELAY_GROUP",
    "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON", "TAXI_IN",
    "ARR_TIME", "ARR_DELAY", "ARR_DEL15", "ARR_DELAY_NEW", "ARR_DELAY_GROUP",
    "ACTUAL_ELAPSED_TIME", "AIR_TIME",
    "CANCELLED", "CANCELLATION_CODE", "DIVERTED",
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
    "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME"
]
found_leakage = [c for c in leakage_candidates if c in df_12m.columns]
print(f"Leakage columns to exclude ({len(found_leakage)}):")
for c in found_leakage:
    print(f"  - {c}")

## Correlation Analysis

In [0]:
corr_cols = [
    "DEP_DEL15",
    "DISTANCE", "CRS_ELAPSED_TIME", "CRS_DEP_TIME", "CRS_ARR_TIME",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyWindSpeed", "HourlyWindGustSpeed", "HourlyWindDirection",
    "HourlyVisibility", "HourlyPrecipitation",
    "HourlyRelativeHumidity", "HourlyStationPressure", "HourlySeaLevelPressure"
]
existing_corr = [c for c in corr_cols if c in df_12m.columns]

df_corr = (
    df_12m.select([col(c).cast("double").alias(c) for c in existing_corr])
    .sample(fraction=0.1, seed=42)
    .toPandas()
    .dropna()
)

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(existing_corr)))
ax.set_yticks(range(len(existing_corr)))
ax.set_xticklabels(existing_corr, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(existing_corr, fontsize=8)
for i in range(len(existing_corr)):
    for j in range(len(existing_corr)):
        ax.text(j, i, f"{corr_matrix.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
plt.colorbar(im, ax=ax, label="Pearson Correlation")
plt.title("Correlation Heatmap (12-Month OTPW, 10% sample)")
plt.tight_layout()
plt.show()

### Correlation Observations

**Notable correlations:**
- **CRS_DEP_TIME ↔ CRS_ARR_TIME (0.77):** Highly correlated: flights that depart later also arrive later. We may consider using only one of these to avoid redundancy.
- **DISTANCE ↔ CRS_ELAPSED_TIME (0.99):** Highly correlated: longer flights take more time. One of these is redundant and can be dropped.
- **HourlyDryBulbTemperature ↔ HourlyDewPointTemperature (0.76):** Highly correlated: warm air holds more moisture. Both carry similar information.
- **HourlyStationPressure ↔ HourlySeaLevelPressure (0.37):** Moderate correlated: These are related but not redundant since station pressure depends on elevation.
- **HourlyRelativeHumidity ↔ HourlyVisibility (-0.54):** Moderate correlated: Expected inverse relationship, high humidity reduces visibility.
- **HourlyWindSpeed ↔ HourlyWindGustSpeed (0.54):** Moderate correlated: gusts accompany sustained winds but add information about turbulence.
- **Weak individual correlations with target** (max 0.17) confirm that delays are driven by combinations of features, not single variables

**Implications for modeling:**
- DISTANCE and CRS_ELAPSED_TIME are nearly identical — keep DISTANCE as it is more interpretable.
- CRS_DEP_TIME and CRS_ARR_TIME are highly correlated — keeping only CRS_DEP_TIME since we are predicting departure delay.
- Weather features have moderate inter-correlations but each captures a distinct aspect of conditions, retain all for now and let regularization handle redundancy.

## EDA Summary

### Data Quality
- 3-Month OTPW: 1,401,363 rows x 216 columns; 12-Month OTPW: 5,811,854
 rows x 216 columns
- ~100+ columns have >50% missing data (Daily/Monthly weather aggregates), these will be dropped
- Filtering to FM-15 report type retains 87.3% of flights with substantially complete hourly weather data
- Cancelled flights (3.10% for 3 month and 1.54% for 12 months) removed, they have no departure delay to predict
- No duplicate rows in either dataset

### Target Variable
- DEP_DEL15 class split: ~80% on-time / ~20% delayed, consistent across both datasets
- Modeling will prioritize recall over accuracy due to class imbalance

### Feature Findings
- **Strongest delay signals:** Precipitation (+121.73%), humidity (+6.19%), wind speed (+3.96%), visibility (-2.28%), temperature (-2.77%)
- Delays peak in evening hours and during summer/winter months
- Significant difference by carrier and airport: strong categorical predictors
- **Highly correlated pairs to address:** DISTANCE ↔ CRS_ELAPSED_TIME (0.99), CRS_DEP_TIME ↔ CRS_ARR_TIME (0.77), DryBulb ↔ DewPoint (0.76)
- **Weak individual correlations with target** (max 0.17) confirm that delays are driven by combinations of features, not single variables

### Features Selected for Modeling
- **Numeric:** DISTANCE, CRS_DEP_TIME, and hourly weather features (HourlyDryBulbTemperature, HourlyDewPointTemperature, HourlyWindSpeed, HourlyWindGustSpeed, HourlyWindDirection, HourlyVisibility, HourlyPrecipitation, HourlyRelativeHumidity, HourlyStationPressure, HourlySeaLevelPressure)
- **Categorical:** OP_UNIQUE_CARRIER, ORIGIN, DEST, DAY_OF_WEEK, MONTH, QUARTER, DAY_OF_MONTH, ORIGIN_STATE_ABR
- **Dropped:** CRS_ELAPSED_TIME (redundant with DISTANCE), CRS_ARR_TIME (redundant with CRS_DEP_TIME)
- **Excluded (leakage):** 25+ columns only available after departure

**Important note on null values:** Per the [NOAA LCD documentation](https://www.ncei.noaa.gov/pub/data/cdo/documentation/LCD_documentation.pdf), null values in FM-15 weather fields (e.g., precipitation, wind gust) represent "none observed", meaning 0, not missing data. For example, a null in HourlyPrecipitation means no precipitation was recorded that hour. During data preparation phase of modeling part, **FM-15 weather nulls will be imputed as 0** rather than dropped or treated as unknown.